# Notebook 3 — Anomaly & Change-Point Detection

**Goal:** Detect the exact cycle at which each engine transitions from a
**Healthy** state to an **Impaired** state, providing an early warning signal
for maintenance scheduling.

**Methods used:**
- CUSUM (Cumulative Sum): single earliest change-point per engine
- PELT (Pruned Exact Linear Time via `ruptures`): multi-breakpoint detection


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ruptures as rpt

from src.data_loader  import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor import full_preprocess_pipeline, add_piecewise_rul
from src.changepoint  import (cusum_detector, detect_health_transitions,
                               classify_health_state)

datasets = load_all_datasets(data_dir='../data/raw')

# Run preprocessing (or load from processed if NB02 was already run)
import os
scalers = {}
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    df_tr, df_te, scaler = full_preprocess_pipeline(
        df_train=datasets[ds_id]['train'],
        df_test=datasets[ds_id]['test'],
        feature_cols=FEATURE_COLS,
        sensor_cols=SENSOR_COLS,
        smooth=True, max_rul=125,
        scaler_save_path=f'../models/saved/scaler_{ds_id}.joblib'
    )
    datasets[ds_id]['train_norm'] = df_tr
    datasets[ds_id]['test_norm']  = df_te
    scalers[ds_id] = scaler

print("Preprocessing complete.")


## 3.1 CUSUM Detector Demonstration on Single Engine


In [ ]:
df_fd001 = datasets['FD001']['train_norm']
unit_1   = df_fd001[df_fd001['unit_id'] == 1].sort_values('cycle')

fig, axes = plt.subplots(len(['sensor_11', 'sensor_12', 'sensor_14']), 1,
                          figsize=(14, 12))
for ax, sensor in zip(axes, ['sensor_11', 'sensor_12', 'sensor_14']):
    cp_idx = cusum_detector(unit_1[sensor].values, threshold=5.0, drift=0.5)
    cp_cycle = unit_1['cycle'].iloc[cp_idx] if cp_idx is not None else None

    ax.plot(unit_1['cycle'], unit_1[sensor], color='steelblue',
            linewidth=1.5, label=sensor)
    if cp_cycle is not None:
        ax.axvline(cp_cycle, color='red', linestyle='--', linewidth=2,
                   label=f'CUSUM trigger @ cycle {cp_cycle}')
        ax.fill_betweenx(
            [unit_1[sensor].min(), unit_1[sensor].max()],
            cp_cycle, unit_1['cycle'].max(),
            alpha=0.08, color='red'
        )
        ax.text(cp_cycle + 1, unit_1[sensor].mean(),
                'IMPAIRED', color='red', fontsize=9, fontstyle='italic')
    ax.axvspan(0, cp_cycle if cp_cycle else unit_1['cycle'].max(),
               alpha=0.04, color='green')
    ax.text(5, unit_1[sensor].max() * 0.95, 'HEALTHY',
            color='green', fontsize=9, fontstyle='italic')
    ax.set_title(f'{sensor} — Engine Unit 1 (FD001)')
    ax.set_xlabel('Cycle')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('CUSUM Health State Transition Detection — Unit 1, FD001', fontsize=13)
plt.tight_layout()
plt.show()


**Insight:** CUSUM detects the transition cycle for each sensor independently.
The earliest detection across all sensors defines the engine's health
transition point. This two-zone view (green=Healthy, red=Impaired) is the
key output for factory operators: it tells them exactly when to start
monitoring this engine more closely.


## 3.2 Fleet-Wide Health Transition Statistics (FD001)


In [ ]:
transitions = detect_health_transitions(df_fd001, SENSOR_COLS, threshold=5.0)
print(transitions.describe().round(1))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(transitions['health_transition_cycle'], bins=20,
             color='coral', edgecolor='black', alpha=0.8)
axes[0].set_xlabel('Cycle of Health Transition')
axes[0].set_ylabel('Count')
axes[0].set_title('When Does Impairment Begin?\n(FD001 Fleet)')
axes[0].axvline(transitions['health_transition_cycle'].median(), color='red',
                linestyle='--', label=f"Median: {transitions['health_transition_cycle'].median():.0f}")
axes[0].legend()

axes[1].hist(transitions['rul_at_transition'], bins=20,
             color='steelblue', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('RUL at Transition Point (cycles remaining)')
axes[1].set_ylabel('Count')
axes[1].set_title('How Much Warning Does CUSUM Provide?')
axes[1].axvline(transitions['rul_at_transition'].median(), color='navy',
                linestyle='--',
                label=f"Median: {transitions['rul_at_transition'].median():.0f} cycles")
axes[1].legend()

axes[2].scatter(transitions['max_cycle'], transitions['health_transition_cycle'],
                alpha=0.6, color='purple', s=40)
axes[2].plot([0, 400], [0, 400], 'r--', linewidth=1, alpha=0.5, label='y=x')
axes[2].set_xlabel('Total Engine Life (cycles)')
axes[2].set_ylabel('Change-Point Cycle')
axes[2].set_title('Change-Point vs. Engine Lifetime')
axes[2].legend()

plt.suptitle('Fleet-Wide CUSUM Health Transition Analysis — FD001', fontsize=13)
plt.tight_layout()
plt.show()


**Insight:** The median CUSUM warning lead time represents how many cycles
in advance operators can be alerted to schedule maintenance. If this is
substantially above 0, the system provides actionable advance notice.
Engines with longer total lives tend to show later change-points,
suggesting consistent degradation proportional to operation duration.


## 3.3 PELT Multi-Breakpoint Detection (FD003 — Two Fault Modes)

FD003 has TWO fault modes (HPC degradation AND Fan degradation), which may
produce multiple structural breaks. PELT detects all of them.


In [ ]:
df_fd003 = datasets['FD003']['train_norm']
sample_units = [1, 5, 10]

for unit_id in sample_units:
    unit_data = df_fd003[df_fd003['unit_id'] == unit_id].sort_values('cycle')
    signal    = unit_data[['sensor_11', 'sensor_12', 'sensor_14']].values

    model      = rpt.Pelt(model='rbf').fit(signal)
    breakpoints = model.predict(pen=3)

    fig, ax = plt.subplots(figsize=(14, 5))
    for col, color in zip(['sensor_11', 'sensor_12', 'sensor_14'],
                           ['steelblue', 'coral', 'seagreen']):
        ax.plot(unit_data['cycle'].values, unit_data[col].values,
                label=col, color=color, alpha=0.8)

    colors_bp = ['red', 'purple', 'orange', 'brown']
    for i, bp in enumerate(breakpoints[:-1]):
        bp_cycle = unit_data['cycle'].iloc[min(bp, len(unit_data)-1)]
        ax.axvline(bp_cycle, color=colors_bp[i % len(colors_bp)],
                   linestyle='--', linewidth=2,
                   label=f'PELT breakpoint #{i+1} @ cycle {bp_cycle}')

    ax.set_title(f'PELT Multi-Breakpoint Detection — Unit {unit_id}, FD003 (2 fault modes)')
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Normalised Sensor Value')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


**Insight:** PELT reveals the full degradation phase structure. In FD003
with two fault modes, some engines show two distinct breakpoints — one for
the onset of Fan degradation and another for HPC degradation acceleration.
This multi-phase awareness is valuable for machines with known multi-stage
wear mechanisms.


## 3.4 Health State Label Summary Table


In [ ]:
def build_health_summary(datasets_dict):
    """Summarise health state detection across all four datasets."""
    records = []
    for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
        df    = datasets_dict[ds_id]['train_norm']
        trans = detect_health_transitions(df, SENSOR_COLS, threshold=5.0)
        records.append({
            'Dataset':                 ds_id,
            'Engines':                 len(trans),
            'Median Transition Cycle': trans['health_transition_cycle'].median().round(1),
            'Median RUL at Warning':   trans['rul_at_transition'].median().round(1),
            'Min RUL at Warning':      trans['rul_at_transition'].min(),
            'Engines With Warning>20': (trans['rul_at_transition'] > 20).sum()
        })
    return pd.DataFrame(records)

summary = build_health_summary(datasets)
print(summary.to_string(index=False))


**Insight:** "Median RUL at Warning" is the key maintenance value metric —
it quantifies how many cycles of advance notice the CUSUM system provides
across the fleet. Higher values mean more time to plan and execute maintenance.
"Engines with Warning > 20 cycles" shows the practical coverage of the system.
